# MÁSTER BIG DATA & DATA ENGINEERING
## Programación Avanzada en Python — Trabajo Final

### ETAPA 1: Análisis exploratorio de multas de tráfico de Madrid (diciembre 2024)

## 0. Importación de librerías

Importamos todas las librerías necesarias al inicio del notebook, siguiendo el estilo PEP-8.

In [ ]:
import io
import re
import urllib
from os import listdir

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

## 1. Descarga y carga de datos

Descargamos los datos de multas de diciembre de 2024 desde el portal de datos abiertos del Ayuntamiento de Madrid usando `requests`.

In [ ]:
# URL del fichero CSV de multas de diciembre 2024
url = "https://datos.madrid.es/egob/catalogo/210104-395-multas-circulacion-detalle.csv"

# Realizamos la petición GET
response = requests.get(url)

# Comprobamos que la descarga fue correcta (código 200 = OK)
if response.status_code == 200:
    print(f"Descarga correcta. URL final tras redirección: {response.url}")
else:
    print(f"Error en la descarga. Código de estado: {response.status_code}")

Como puede observarse, el servidor redirige la URL original a la URL definitiva del fichero CSV.

Ahora leemos el contenido con `pandas.read_csv`, usando el separador `;` y la codificación `latin1`:

In [ ]:
# Convertimos el texto de la respuesta HTTP a un objeto tipo fichero en memoria
content = io.StringIO(response.text)

# Leemos el CSV con separador ; y codificación latin1
multas = pd.read_csv(content, sep=';', encoding='latin1')

# Mostramos las primeras filas para verificar la carga
multas.head()

## 2. Análisis exploratorio

### 2.1 Estructura general del DataFrame (`df.info()`)

In [ ]:
# Información general del DataFrame
multas.info()

**Comentarios sobre el resultado de `info()`:**

- El DataFrame tiene 14 columnas.
- La mayoría de columnas son de tipo `object` (texto), excepto `MES`, `ANIO`, `PUNTOS` e `IMP_BOL` que son numéricas.
- `HORA` aparece como `float64` ya que representa la hora en formato decimal (p.ej. `20.23` → 20h 23min).
- Las columnas `VEL_LIMITE`, `VEL_CIRCULA`, `COORDENADA-X` y `COORDENADA-Y` aparecen como `object` pese a ser numéricas, probablemente por contener espacios en blanco o celdas vacías.
- Hay columnas con muchos valores nulos (`VEL_LIMITE`, `VEL_CIRCULA`, `COORDENADA-X`, `COORDENADA-Y`), ya que muchas multas no son por exceso de velocidad ni tienen coordenadas registradas.

In [ ]:
# Número de filas y columnas
print(f"Número de filas: {multas.shape[0]}")
print(f"Número de columnas: {multas.shape[1]}")

In [ ]:
# Tipos de datos de cada columna
print("Tipos de datos:")
print(multas.dtypes)

In [ ]:
# Valores no nulos por columna
print("Valores no nulos por columna:")
print(multas.notnull().sum())

## 3. Limpieza de datos

### 3a. Columnas categóricas: eliminar espacios en blanco

In [ ]:
# a) Columna CALIFICACION: número de valores únicos ANTES de limpiar
print("Valores únicos en CALIFICACION (antes):", multas['CALIFICACION'].nunique())
print(multas['CALIFICACION'].unique())

# Eliminar espacios en blanco al inicio y final
multas['CALIFICACION'] = multas['CALIFICACION'].str.strip()

# Número de valores únicos DESPUÉS de limpiar
print("\nValores únicos en CALIFICACION (después):", multas['CALIFICACION'].nunique())
print(multas['CALIFICACION'].unique())

In [ ]:
# b) Columnas DESCUENTO, HECHO-BOL y DENUNCIANTE
for col in ['DESCUENTO', 'HECHO-BOL', 'DENUNCIANTE']:
    print(f"Valores únicos en {col} (antes): {multas[col].nunique()}")
    multas[col] = multas[col].str.strip()
    print(f"Valores únicos en {col} (después): {multas[col].nunique()}")
    print()

### 3b. Nombres de columnas: detectar y eliminar espacios

In [ ]:
# c) Mostrar nombres de columnas actuales
print("Nombres de columnas (con posibles espacios):")
for col in multas.columns:
    print(repr(col))  # repr() muestra los espacios invisibles

In [ ]:
# Detectar columnas con espacios al final
cols_con_espacios = [col for col in multas.columns if col != col.strip()]
print(f"Columnas con espacios: {cols_con_espacios}")

# Mostrar la columna VEL_CIRCULA (puede tener espacio al final)
print("\nColumna de velocidad de circulación:", repr([c for c in multas.columns if 'VEL_CIRCULA' in c][0]))

# Mostrar la columna COORDENADA-Y (puede tener espacios al final)
print("Columna COORDENADA-Y:", repr([c for c in multas.columns if 'COORDENADA' in c and 'Y' in c][0]))

In [ ]:
# Serie de datos correspondiente a COORDENADA-Y (antes de renombrar)
col_coord_y = [c for c in multas.columns if 'COORDENADA' in c and 'Y' in c][0]
print(f"Serie COORDENADA-Y (columna: {repr(col_coord_y)}):")
multas[col_coord_y].head(10)

In [ ]:
# Renombrar columnas: eliminar espacios al inicio y final de todos los nombres
multas.rename(columns=lambda x: x.strip(), inplace=True)

print("Nombres de columnas (tras limpiar):")
for col in multas.columns:
    print(repr(col))

### 3c. Columnas de velocidad: conversión a numérico

In [ ]:
# d) Tipo actual de las columnas de velocidad
print("Tipo actual de VEL_LIMITE:", multas['VEL_LIMITE'].dtype)
print("Tipo actual de VEL_CIRCULA:", multas['VEL_CIRCULA'].dtype)

# Valores únicos (puede haber strings vacíos o con espacios)
print("\nValores únicos en VEL_LIMITE:", multas['VEL_LIMITE'].nunique())
print(multas['VEL_LIMITE'].unique()[:10])

print("\nValores únicos en VEL_CIRCULA:", multas['VEL_CIRCULA'].nunique())
print(multas['VEL_CIRCULA'].unique()[:10])

In [ ]:
# Convertir a numérico: errors='coerce' convierte los valores no numéricos a NaN
multas['VEL_LIMITE'] = pd.to_numeric(multas['VEL_LIMITE'], errors='coerce')
multas['VEL_CIRCULA'] = pd.to_numeric(multas['VEL_CIRCULA'], errors='coerce')

print("Tipo de VEL_LIMITE tras conversión:", multas['VEL_LIMITE'].dtype)
print("Tipo de VEL_CIRCULA tras conversión:", multas['VEL_CIRCULA'].dtype)

### 3d. Coordenadas: conversión a numérico

In [ ]:
# e) Tipo actual de las columnas de coordenadas
print("Tipo actual de COORDENADA-X:", multas['COORDENADA-X'].dtype)
print("Tipo actual de COORDENADA-Y:", multas['COORDENADA-Y'].dtype)

# Convertir a numérico
multas['COORDENADA-X'] = pd.to_numeric(multas['COORDENADA-X'], errors='coerce')
multas['COORDENADA-Y'] = pd.to_numeric(multas['COORDENADA-Y'], errors='coerce')

print("\nTipo de COORDENADA-X tras conversión:", multas['COORDENADA-X'].dtype)
print("Tipo de COORDENADA-Y tras conversión:", multas['COORDENADA-Y'].dtype)

## 4. Crear columna fecha

### a) Crear la columna `fecha` combinando ANIO, MES y HORA

La columna HORA está en formato decimal (p.ej. `20.23` = 20h 23min). Usamos día fijo = 1.

In [ ]:
# Extraer la parte entera (hora) y la parte decimal (minutos)
hora_decimal = pd.to_numeric(multas['HORA'], errors='coerce').fillna(0)
hora_int = hora_decimal.astype(int)
minuto_int = ((hora_decimal - hora_int) * 100).round().astype(int)

# Crear la columna fecha usando pd.to_datetime
multas['fecha'] = pd.to_datetime(
    {
        'year': multas['ANIO'],
        'month': multas['MES'],
        'day': 1,
        'hour': hora_int,
        'minute': minuto_int,
    },
    errors='coerce'
)

print("Tipo de la columna fecha:", multas['fecha'].dtype)
print(multas[['MES', 'ANIO', 'HORA', 'fecha']].head())

### b) Establecer `fecha` como índice del DataFrame

In [ ]:
# Establecer la columna fecha como índice
multas.set_index('fecha', inplace=True)

print("Índice del DataFrame:", multas.index.dtype)
multas.head(3)

### c) Eliminar la columna `fecha` auxiliar

Como ya hemos establecido `fecha` como índice, si existiera como columna la eliminaríamos. En pandas, `set_index` ya la elimina automáticamente, pero por completitud:

In [ ]:
# Eliminar la columna 'fecha' si aún existiera como columna (por seguridad)
if 'fecha' in multas.columns:
    multas.drop(columns=['fecha'], inplace=True)

print("Columnas finales:")
print(list(multas.columns))
print("\nÍndice:")
print(multas.index[:5])

## 5. Verificación del resultado final

In [ ]:
# Resumen final del DataFrame limpio
print("=" * 60)
print("RESUMEN FINAL DEL DATAFRAME LIMPIO")
print("=" * 60)
print(f"Filas: {multas.shape[0]}")
print(f"Columnas: {multas.shape[1]}")
print()
multas.info()
print()
multas.head()